# 01.4 — Serialization and JSON Schema

[Previous: 01.3](01.3-schemas-and-validation.ipynb) · [Class index and setup](README.md) · [Next: 01.5](01.5-practice-and-review.ipynb)

**Purpose:** Distinguish a record, its exported data, and the contract describing records.

**In class:** 0 minutes. **Independent work:** 45 minutes, included in the class index's 210-minute allowance. The separately assessed project adds workload.

## What you will be able to do

- Convert a validated instance to a dictionary and JSON text.
- Validate a JSON round trip without confusing equality with identity.
- Read required fields and constraints in a generated JSON Schema.

## Starting point

An Observation validates a label and a nonnegative quantity. Its complete definition is included below. You do not need an earlier report or exercise answer.

This file includes all definitions and inputs it needs. Read the parts in order, but start this one with **Restart Kernel → Run All** in VS Code. Do not execute another notebook or an earlier exercise solution to prepare it. Empty answer cells are safe during Run All.

Reference solutions stay in separate cells: attempt the task, uncomment its `%load` line, and run that cell to reveal source. Inspect the result before running the populated cell separately. Keep your answer in its own cell because loading can replace the invoking cell. A loaded solution remains in the notebook even after restarting the kernel.

In [ ]:
from pathlib import Path
import sys

if not Path("solutions/exercise_01_4_h1.py").is_file():
    raise FileNotFoundError(
        "Open 01-oop-and-schemas.code-workspace in VS Code and restart the kernel. "
        "The working directory must be the class folder containing README.md and solutions/."
    )
print("Notebook kernel:", sys.executable)
print("Class files are available.")

## Reading and demonstrations — Independent work · 20 minutes

This part's 45 minutes comprise 20 minutes of reading/demos, 15 minutes of exercise, and ten minutes of review. The exercise and review are included, not additional.

### Four different things

| Representation | What it is for |
| --- | --- |
| Observation instance | Validated data with Python attribute access |
| Python dictionary | Ordinary Python data passed to other code |
| JSON text | A text representation that another process can exchange |
| JSON Schema | A description of permitted structure and constraints |

A dictionary printed in a notebook is not automatically JSON text. JSON has its own text syntax. Pydantic can produce either a dictionary or JSON, while its schema method describes the model rather than a particular observation.

In [ ]:
from pydantic import BaseModel, Field, ValidationError


class Observation(BaseModel):
    """A nonempty label and a finite, nonnegative quantity."""
    label: str = Field(min_length=1)
    value: float = Field(ge=0, allow_inf_nan=False)

In [ ]:
import json

observation = Observation(label="north", value=2.5)
as_dictionary = observation.model_dump()
as_json_text = observation.model_dump_json()
print("Dictionary:", as_dictionary, type(as_dictionary).__name__)
print("JSON text:", as_json_text, type(as_json_text).__name__)
assert isinstance(as_dictionary, dict) and isinstance(as_json_text, str)
assert json.loads(as_json_text) == as_dictionary

### Illustration — Serialization

<img src="images/serialization.png" alt="Serialization turns an object into a byte stream for storage or transfer; deserialization reconstructs an object from a stream." width="850" style="max-width: 100%; height: auto; background-color: white;">

Serialization represents data for storage or transfer; deserialization reconstructs data. Here model_dump_json returns JSON text, not bytes; encoding is a separate step when bytes are required. The round trip creates a new validated Observation, rather than preserving the original object identity. The database and memory branches are context, not extra tasks.

*Credit: [Serialization](https://commons.wikimedia.org/wiki/File:Serialization.png) by Trustable, [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/). Commons PNG rendering; no content changes. [Image sources and teaching notes](IMAGE_SUGGESTIONS.md).*

### A round trip validates again

`model_validate_json` parses JSON text and validates it using the record contract. The reconstructed object has equal field values but is a new instance. Compare the data or equality, not object identity. Comparing exact JSON whitespace is also not a useful data-equivalence check.

In [ ]:
restored = Observation.model_validate_json(as_json_text)
print("Equal values:", restored == observation)
print("Same instance:", restored is observation)
assert restored.model_dump() == observation.model_dump()
assert restored is not observation

Malformed text and invalid field values can both prevent a valid record from being constructed. The next cell handles the expected failure so Run All still succeeds. A future API client will use the same boundary pattern after receiving a response.

In [ ]:
for payload in ['{"label":"north","value":-1}', 'this is not JSON']:
    try:
        Observation.model_validate_json(payload)
    except ValidationError as error:
        print("Rejected:", error.errors()[0]["type"])

### Visual guide — Follow one record through its representations

<img src="images/record-representations.png" alt="The Observation class validates a north 2.5 input into an instance; the instance exports a dictionary or JSON text; the class separately generates JSON Schema." width="545" style="max-width: 100%; height: auto; background-color: white;">

An instance stores one validated observation. A dictionary and JSON text represent that observation. JSON Schema describes allowed records and is generated from the model class, rather than from the north record.

*Credit: Original ADPRO course diagram (2026), [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). [Editable Mermaid source](images/record-representations.mmd); [sources and teaching notes](IMAGE_SUGGESTIONS.md).*

### Generate the contract

`model_json_schema()` returns a dictionary describing fields and constraints. `properties` describes each field, `required` lists fields that must be supplied, and this model's numeric `minimum` is zero. Generating the schema does not serialize the instance stored in `observation`.

In [ ]:
schema = Observation.model_json_schema()
print(json.dumps(schema, indent=2))
assert set(schema["required"]) == {"label", "value"}
assert schema["properties"]["label"]["minLength"] == 1
assert schema["properties"]["value"]["minimum"] == 0

This creates a bridge between Python data models and external contracts. In later blocks, an API response, model tool argument, or structured model result can be checked against an explicit shape. You do not need to learn the full JSON Schema specification today.

A schema is also not a guarantee of truth: it cannot establish that north really measured 2.5 units. Some application rules are not represented by a particular schema, and a program still needs to perform its chosen runtime validation.

### 🧩 Exercise 01.4.H1 — Export a record and inspect its contract

> **Independent work · 15 minutes**
>
> **Goal:** Use serialization and schema generation for their different purposes.
>
> **Start here:** The complete Observation definition in this notebook; create a fresh instance in your answer.
>
> **Task:** Export an observation to JSON text, validate that text into a new instance, and check that the data is preserved. Generate its schema; find required fields and the value minimum. Show that a negative quantity is rejected.
>
> **Check:** The round trip preserves label/value; required contains label and value; the minimum is 0. Your explanation distinguishes exported values from the contract.
>
> **Hint:** Use model_dump_json, model_validate_json, and model_json_schema. json.loads can compare JSON text with a dictionary semantically.

In [ ]:
# Your attempt for 01.4.H1.
# Keep your code and explanation here, separate from the reference cell.

In [ ]:
# Reveal only after attempting the exercise.
# Remove the leading '# ' from the next line and run this cell.
# %load solutions/exercise_01_4_h1.py

## Review — Independent work · 10 minutes

Answer these within the remaining time:

1. Which call exports the value 2.5, and which describes the permitted numeric field?
2. Why is the restored object equal in data but not the same instance?
3. Why should a consumer validate again even if the producer claims to use a schema?
4. Give one false measurement that would satisfy this structural contract.

**Completion evidence:** a JSON round trip, the two inspected schema properties, an expected validation failure, and a short explanation of the contract's limits.

**Optional extension · 15 minutes, excluded from required time:** add a field with a default and inspect whether it appears in `required`. The practice file also provides an opportunity to explore defaults.

## References

- [Pydantic serialization](https://docs.pydantic.dev/latest/concepts/serialization/)
- [Pydantic JSON Schema](https://docs.pydantic.dev/latest/concepts/json_schema/)
- [Python JSON](https://docs.python.org/3/library/json.html)
- [Previous: schemas and validation](01.3-schemas-and-validation.ipynb)
- [Next: practice and review](01.5-practice-and-review.ipynb)
- [Class index](README.md)